# attempt to use API directly?

In [3]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import json
from pandas import json_normalize

import os
import warnings
from urllib.parse import urlencode, urljoin
from urllib.request import urlopen

from cartopy import crs as ccrs, feature as cfeature
import netCDF4
from netCDF4 import Dataset
from datetime import datetime as dt

import xarray as xr

## using pointers from this page: https://eccc-msc.github.io/open-data/msc-geomet/ogc_api_en/ and https://pavics-sdi.readthedocs.io/en/latest/notebooks/eccc-geoapi-xclim.html, and collection documentation from https://api.weather.gc.ca/collections

## Collection: climate-daily

In [3]:
# Compose the request
host = "https://api.weather.gc.ca/"
api = urljoin(host, "collections/climate-daily/items/")
query = {
    "datetime": "2026-10-13/..", # all items from 2026-10-05 onward
    #"datetime": "1840-03-01 00:00:00/2021-06-02 00:00:00",
    "bbox": "-123.25,49,-123.0,49.25",# minX, minY, maxX, maxY
    #"STN_ID": "10761",
    "sortby": "LOCAL_DATE",
    "f": "csv"
}
url = f"{api}?{urlencode(query)}"


# Use geopandas to convert the json output to a GeoDataFrame.
with urlopen(url=str(url)) as req:
    df = pd.read_csv(req)
df.sort_index().head(20)

,x,y,STATION_NAME,CLIMATE_IDENTIFIER,ID,LOCAL_DATE,PROVINCE_CODE,LOCAL_YEAR,LOCAL_MONTH,LOCAL_DAY,...,DIRECTION_MAX_GUST_FLAG,SPEED_MAX_GUST,SPEED_MAX_GUST_FLAG,COOLING_DEGREE_DAYS,COOLING_DEGREE_DAYS_FLAG,HEATING_DEGREE_DAYS,HEATING_DEGREE_DAYS_FLAG,MIN_REL_HUMIDITY,MIN_REL_HUMIDITY_FLAG,STN_ID
0,-123.000000,49.216667,BURNABY METROTOWN,110A1ND,110A1ND.2026.10.13,2026-10-13 00:00:00,BC,2026,10,13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,671
1,-123.000000,49.250000,BURNABY BC INST TECHNOL,1101142,1101142.2026.10.13,2026-10-13 00:00:00,BC,2026,10,13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,722
2,-123.016667,49.216667,BURNABY SUNCREST,1101160,1101160.2026.10.13,2026-10-13 00:00:00,BC,2026,10,13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,732
3,-123.066667,49.033333,DELTA BEACH GROVE,1102412,1102412.2026.10.13,2026-10-13 00:00:00,BC,2026,10,13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,759
4,-123.066667,49.083333,DELTA LADNER EAST,1102416,1102416.2026.10.13,2026-10-13 00:00:00,BC,2026,10,13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,760
5,-123.083333,49.066667,DELTA LADNER SOUTH,1102417,1102417.2026.10.13,2026-10-13 00:00:00,BC,2026,10,13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,761
6,-123.083333,49.000000,DELTA PEBBLE HILL,1102420,1102420.2026.10.13,2026-10-13 00:00:00,BC,2026,10,13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,763
7,-123.100000,49.016667,DELTA TSAWWASSEN,1102424,1102424.2026.10.13,2026-10-13 00:00:00,BC,2026,10,13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,765
8,-123.093333,49.010944,DELTA TSAWWASSEN BEACH,1102425,1102425.2026.10.13,2026-10-13 00:00:00,BC,2026,10,13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,766
9,-123.066667,49.083333,LADNER,1104468,1104468.2026.10.13,2026-10-13 00:00:00,BC,2026,10,13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,789


## Collection: climate-normals

In [4]:
# Compose the request
host = "https://api.weather.gc.ca/"
api = urljoin(host, "collections/climate-normals/items/")
query = {
    #"DATE_CALCULATED" : "2026-10-05",
    #"datetime": "2026-10-05", # all items from 2026-10-05 onward
    #"datetime": "1840-03-01 00:00:00/2021-06-02 00:00:00",
    #"bbox": "-123.25,49,-123.0,49.25",# minX, minY, maxX, maxY
    "STN_ID": "24", # Vancouver Intl A
    "properties": "VALUE",
    #"sortby": "LOCAL_DATE",
    "f": "csv"
}
url = f"{api}?{urlencode(query)}"


# Use pandas to convert the csv output to a data frame.
with urlopen(url=str(url)) as req:
    df = pd.read_csv(req)
df.sort_index().head()

,x,y,VALUE
0,-123.392128,48.573342,0.0
1,-123.392128,48.573342,0.0
2,-123.392128,48.573342,0.0
3,-123.392128,48.573342,0.0
4,-123.392128,48.573342,0.0


#### huh that didn't work - wrong station?

## Collection: climate-stations

In [15]:
# Compose the request
host = "https://api.weather.gc.ca/"
api = urljoin(host, "collections/climate-stations/items/")
query = {"ENG_PROV_NAME": "BRITISH-COLUMBIA",
         "limit":"10000",
        "f":"csv"}
url = f"{api}?{urlencode(query)}"
print(url)

# Send the request to the server
resp = requests.get(url)
resp

#url = f"{api}?{urlencode(query)}"


# Use pandas to convert the csv output to a data frame.
with urlopen(url=str(url)) as req:
    df = pd.read_csv(req)
df.sort_index().head(100)

https://api.weather.gc.ca/collections/climate-stations/items/?ENG_PROV_NAME=BRITISH-COLUMBIA&limit=10000&f=csv


,x,y,STN_ID,STATION_NAME,PROV_STATE_TERR_CODE,ENG_PROV_NAME,FRE_PROV_NAME,COUNTRY,LATITUDE,LONGITUDE,...,LAST_DATE,HLY_FIRST_DATE,HLY_LAST_DATE,DLY_FIRST_DATE,DLY_LAST_DATE,MLY_FIRST_DATE,MLY_LAST_DATE,HAS_MONTHLY_SUMMARY,HAS_NORMALS_DATA,HAS_HOURLY_DATA
0,-123.700000,48.916667,2,CHEMAINUS BARE POINT,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,485500000,-1234200000,...,1979-12-31 00:00:00,NaN,NaN,1979-11-01 00:00:00,1979-12-31 00:00:00,1979-01-01 00:00:00,1979-12-01 00:00:00,Y,N,N
1,-123.350000,48.883333,3,GALIANO SOUTH 2,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,485300000,-1232100000,...,1984-12-01 00:00:00,NaN,NaN,1979-05-01 00:00:00,1984-01-31 00:00:00,1979-01-01 00:00:00,1984-12-01 00:00:00,Y,N,N
2,-123.316667,48.866667,4,GALIANO SOUTH,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,485200000,-1231900000,...,1978-12-31 00:00:00,NaN,NaN,1977-07-01 00:00:00,1978-12-31 00:00:00,1977-01-01 00:00:00,1978-12-01 00:00:00,Y,N,N
3,-123.416667,48.550000,5,SAANICH OLDFIELD NORTH,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,483300000,-1232500000,...,1967-12-01 00:00:00,NaN,NaN,1965-12-01 00:00:00,1967-03-31 00:00:00,1966-01-01 00:00:00,1967-12-01 00:00:00,Y,N,N
4,-123.266944,48.455000,6,VICTORIA PHYLLIS STREET,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,482718000,-1231601000,...,2001-08-31 00:00:00,NaN,NaN,1973-06-01 00:00:00,2001-08-31 00:00:00,1973-01-01 00:00:00,2001-08-01 00:00:00,Y,N,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,-123.626408,48.646947,97,SHAWNIGAN LAKE,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,483849010,-1233735070,...,2023-11-02 00:00:00,NaN,NaN,1911-05-01 00:00:00,2023-11-02 00:00:00,1911-01-01 00:00:00,2007-02-01 00:00:00,Y,Y,N
96,-123.633333,48.650000,98,SHAWNIGAN LAKE NORTH,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,483900000,-1233800000,...,1976-12-01 00:00:00,NaN,NaN,1973-02-01 00:00:00,1976-04-30 00:00:00,1973-01-01 00:00:00,1976-12-01 00:00:00,Y,N,N
97,-123.400000,48.683333,99,SIDNEY DEEP COVE,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,484100000,-1232400000,...,1973-12-01 00:00:00,NaN,NaN,1967-04-01 00:00:00,1973-05-31 00:00:00,1967-01-01 00:00:00,1973-12-01 00:00:00,Y,N,N
98,-123.733333,48.366667,100,SOOKE,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,482200000,-1234400000,...,1992-12-01 00:00:00,NaN,NaN,1970-03-01 00:00:00,1992-06-30 00:00:00,1970-01-01 00:00:00,1992-12-01 00:00:00,Y,N,N


In [16]:
df.query("STATION_NAME.str.contains('VANCOUVER')")

,x,y,STN_ID,STATION_NAME,PROV_STATE_TERR_CODE,ENG_PROV_NAME,FRE_PROV_NAME,COUNTRY,LATITUDE,LONGITUDE,...,LAST_DATE,HLY_FIRST_DATE,HLY_LAST_DATE,DLY_FIRST_DATE,DLY_LAST_DATE,MLY_FIRST_DATE,MLY_LAST_DATE,HAS_MONTHLY_SUMMARY,HAS_NORMALS_DATA,HAS_HOURLY_DATA
670,-123.050000,49.316667,676,N VANCOUVER CLOVERLEY,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,491900000,-1230300000,...,1984-12-01 00:00:00,NaN,NaN,1968-07-01 00:00:00,1984-04-30 00:00:00,1968-01-01 00:00:00,1984-12-01 00:00:00,Y,N,N
672,-123.016667,49.316667,678,N VANCOUVER SEYMOUR BLVD,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,491900000,-1230100000,...,1982-12-01 00:00:00,NaN,NaN,1968-01-01 00:00:00,1982-03-31 00:00:00,1968-01-01 00:00:00,1982-12-01 00:00:00,Y,N,N
673,-123.116667,49.350000,679,N VANCOUVER CAPILANO HIGHLANDS,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,492100000,-1230700000,...,1991-11-01 00:00:00,NaN,NaN,1989-04-01 00:00:00,1991-09-30 00:00:00,1989-01-01 00:00:00,1991-11-01 00:00:00,Y,N,N
674,-123.107222,49.359722,680,N VANCOUVER CLEVELAND,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,492135000,-1230626000,...,2003-09-30 00:00:00,NaN,NaN,1968-04-01 00:00:00,2003-09-30 00:00:00,1968-01-01 00:00:00,2003-09-01 00:00:00,Y,N,N
682,-123.050000,49.283333,688,VANCOUVER GRANDVIEW N,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,491700000,-1230300000,...,1963-07-31 00:00:00,NaN,NaN,1961-05-01 00:00:00,1963-07-31 00:00:00,1961-01-01 00:00:00,1962-12-01 00:00:00,Y,N,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1707,-123.187236,49.182500,51357,VANCOUVER SEA ISLAND CCG,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,491057000,-1231114050,...,2026-10-18 12:30:11,2013-03-20 06:00:00,2026-10-18 12:30:11,2013-04-25 00:00:00,2026-10-18 00:00:00,NaN,NaN,Y,N,Y
1712,-123.183889,49.194722,51442,VANCOUVER INTL A,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,491141000,-1231102000,...,2026-10-18 12:30:11,2013-06-11 06:00:00,2026-10-18 12:30:11,2013-06-13 00:00:00,2026-10-18 00:00:00,NaN,NaN,Y,N,Y
1725,-123.183889,49.194722,52218,VANCOUVER INT'L HWOS,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,491141000,-1231102000,...,2014-07-04 00:00:00,NaN,NaN,2014-02-21 00:00:00,2014-07-04 00:00:00,NaN,NaN,NaN,NaN,NaN
1758,-123.111389,49.294444,55438,VANCOUVER HARBOUR,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,491740000,-1230641000,...,2026-10-18 12:30:11,2023-04-24 09:00:00,2026-10-18 12:30:11,2023-04-24 00:00:00,2026-10-18 00:00:00,NaN,NaN,Y,N,N


In [17]:
# which ones have climate normals?
# help from https://stackoverflow.com/questions/44933071/select-rows-by-partial-string-with-query-with-pandas
df.query("HAS_NORMALS_DATA == 'Y' & STATION_NAME.str.contains('VANCOUVER')")

,x,y,STN_ID,STATION_NAME,PROV_STATE_TERR_CODE,ENG_PROV_NAME,FRE_PROV_NAME,COUNTRY,LATITUDE,LONGITUDE,...,LAST_DATE,HLY_FIRST_DATE,HLY_LAST_DATE,DLY_FIRST_DATE,DLY_LAST_DATE,MLY_FIRST_DATE,MLY_LAST_DATE,HAS_MONTHLY_SUMMARY,HAS_NORMALS_DATA,HAS_HOURLY_DATA
822,-123.014917,49.297944,830,N VANCOUVER 2ND NARROWS,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,491752600,-1230053700,...,2008-11-30 00:00:00,NaN,NaN,1957-11-01 00:00:00,2008-11-30 00:00:00,1957-01-01 00:00:00,2007-02-01 00:00:00,Y,Y,N
825,-123.115278,49.314783,833,N VANCOUVER WHARVES,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,491853220,-1230655000,...,2026-10-18 00:00:00,NaN,NaN,1962-03-01 00:00:00,2026-10-18 00:00:00,1962-01-01 00:00:00,2007-02-01 00:00:00,Y,Y,N
881,-123.181944,49.195000,889,VANCOUVER INT'L A,BC,BRITISH COLUMBIA,COLOMBIE-BRITANNIQUE,CAN,491142000,-1231055000,...,2013-06-13 10:00:00,1953-01-01 00:00:00,2013-06-13 10:00:00,1937-01-01 00:00:00,2013-06-12 00:00:00,1937-01-01 00:00:00,2013-06-01 00:00:00,Y,Y,Y


In [22]:
## maybe can try to above query for climate normals using this filtered list above

In [18]:
# Compose the request
host = "https://api.weather.gc.ca/"
api = urljoin(host, "collections/climate-normals/items/")
query = {
    #"DATE_CALCULATED" : "2026-10-05",
    #"datetime": "2026-10-05", # all items from 2026-10-05 onward
    #"datetime": "1840-03-01 00:00:00/2021-06-02 00:00:00",
    #"bbox": "-123.25,49,-123.0,49.25",# minX, minY, maxX, maxY
    #"STN_ID": "118", # Victoria Intl A
    "STN_ID": "889", # Vancouver Int'l A
    #"properties": "VALUE",
    #"sortby": "LOCAL_DATE",
    "f": "csv"
}
url = f"{api}?{urlencode(query)}"


# Use pandas to convert the csv output to a data frame.
with urlopen(url=str(url)) as req:
    df_norms = pd.read_csv(req)
df_norms.sort_index().head(100)

,x,y,STATION_NAME,CLIMATE_IDENTIFIER,ID,PERIOD,CURRENT_FLAG,NORMAL_CODE,NORMAL_ID,PUBLICATION_CODE,...,YEAR_COUNT_NORMAL_PERIOD,TOTAL_OBS_COUNT,OCCURRENCE_COUNT,MAX_DURATION_MISSING_YEARS,PERCENT_OF_POSSIBLE_OBS,E_NORMAL_ELEMENT_NAME,F_NORMAL_ELEMENT_NAME,MONTH,VALUE,STN_ID
0,-123.181944,49.195,VANCOUVER INT'L A,1108447,889.1.1,NORM,Y,A,1,1,...,30,930,NaN,0,100.00,Mean daily temperature deg C,Température moyenne quotidienne en degree C,1,4.13,889
1,-123.181944,49.195,VANCOUVER INT'L A,1108447,889.1.2,NORM,Y,A,1,1,...,30,847,NaN,0,100.00,Mean daily temperature deg C,Température moyenne quotidienne en degree C,2,4.90,889
2,-123.181944,49.195,VANCOUVER INT'L A,1108447,889.1.3,NORM,Y,A,1,1,...,30,930,NaN,0,100.00,Mean daily temperature deg C,Température moyenne quotidienne en degree C,3,6.88,889
3,-123.181944,49.195,VANCOUVER INT'L A,1108447,889.1.4,NORM,Y,A,1,1,...,30,899,NaN,0,99.89,Mean daily temperature deg C,Température moyenne quotidienne en degree C,4,9.39,889
4,-123.181944,49.195,VANCOUVER INT'L A,1108447,889.1.5,NORM,Y,A,1,1,...,30,929,NaN,0,99.89,Mean daily temperature deg C,Température moyenne quotidienne en degree C,5,12.80,889
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,-123.181944,49.195,VANCOUVER INT'L A,1108447,889.25.5,NORM,Y,A,25,1,...,29,899,NaN,1,96.67,Total degree-days Above 10 deg C,Total des degrés-jours supérieurs à 10 degree C,5,91.35,889
96,-123.181944,49.195,VANCOUVER INT'L A,1108447,889.25.6,NORM,Y,A,25,1,...,30,900,NaN,0,100.00,Total degree-days Above 10 deg C,Total des degrés-jours supérieurs à 10 degree C,6,169.82,889
97,-123.181944,49.195,VANCOUVER INT'L A,1108447,889.25.7,NORM,Y,A,25,1,...,30,930,NaN,0,100.00,Total degree-days Above 10 deg C,Total des degrés-jours supérieurs à 10 degree C,7,246.70,889
98,-123.181944,49.195,VANCOUVER INT'L A,1108447,889.25.8,NORM,Y,A,25,1,...,29,899,NaN,1,96.67,Total degree-days Above 10 deg C,Total des degrés-jours supérieurs à 10 degree C,8,247.83,889


In [20]:
#df_norms.E_NORMAL_ELEMENT_NAME.unique()

In [21]:
df_norms.query("E_NORMAL_ELEMENT_NAME.str.contains('Mean daily temperature')")

,x,y,STATION_NAME,CLIMATE_IDENTIFIER,ID,PERIOD,CURRENT_FLAG,NORMAL_CODE,NORMAL_ID,PUBLICATION_CODE,...,YEAR_COUNT_NORMAL_PERIOD,TOTAL_OBS_COUNT,OCCURRENCE_COUNT,MAX_DURATION_MISSING_YEARS,PERCENT_OF_POSSIBLE_OBS,E_NORMAL_ELEMENT_NAME,F_NORMAL_ELEMENT_NAME,MONTH,VALUE,STN_ID
0,-123.181944,49.195,VANCOUVER INT'L A,1108447,889.1.1,NORM,Y,A,1,1,...,30,930,NaN,0,100.00,Mean daily temperature deg C,Température moyenne quotidienne en degree C,1,4.13,889
1,-123.181944,49.195,VANCOUVER INT'L A,1108447,889.1.2,NORM,Y,A,1,1,...,30,847,NaN,0,100.00,Mean daily temperature deg C,Température moyenne quotidienne en degree C,2,4.90,889
2,-123.181944,49.195,VANCOUVER INT'L A,1108447,889.1.3,NORM,Y,A,1,1,...,30,930,NaN,0,100.00,Mean daily temperature deg C,Température moyenne quotidienne en degree C,3,6.88,889
3,-123.181944,49.195,VANCOUVER INT'L A,1108447,889.1.4,NORM,Y,A,1,1,...,30,899,NaN,0,99.89,Mean daily temperature deg C,Température moyenne quotidienne en degree C,4,9.39,889
4,-123.181944,49.195,VANCOUVER INT'L A,1108447,889.1.5,NORM,Y,A,1,1,...,30,929,NaN,0,99.89,Mean daily temperature deg C,Température moyenne quotidienne en degree C,5,12.80,889
5,-123.181944,49.195,VANCOUVER INT'L A,1108447,889.1.6,NORM,Y,A,1,1,...,30,900,NaN,0,100.00,Mean daily temperature deg C,Température moyenne quotidienne en degree C,6,15.66,889
6,-123.181944,49.195,VANCOUVER INT'L A,1108447,889.1.7,NORM,Y,A,1,1,...,30,930,NaN,0,100.00,Mean daily temperature deg C,Température moyenne quotidienne en degree C,7,17.96,889
7,-123.181944,49.195,VANCOUVER INT'L A,1108447,889.1.8,NORM,Y,A,1,1,...,30,929,NaN,0,99.89,Mean daily temperature deg C,Température moyenne quotidienne en degree C,8,18.00,889
8,-123.181944,49.195,VANCOUVER INT'L A,1108447,889.1.9,NORM,Y,A,1,1,...,29,870,NaN,0,100.00,Mean daily temperature deg C,Température moyenne quotidienne en degree C,9,14.89,889
9,-123.181944,49.195,VANCOUVER INT'L A,1108447,889.1.10,NORM,Y,A,1,1,...,30,930,NaN,0,100.00,Mean daily temperature deg C,Température moyenne quotidienne en degree C,10,10.29,889


### OK, tricky - have to have all the stations stored somewhere because the request to climate-stations names doesn't return everything. Might need to pass a 'limit' parameter to get them all?
- But can get the monthly normals for a station
- Note that month 13 looks to be the mean for the year (average of the monthly normals)
-  Looks like I can also use CQL to get stations matching a certain name, which might work better? https://eccc-msc.github.io/open-data/msc-geomet/ogc_api_en/#ogc-api-advanced-filtering-with-cql2

# So, got the normals! On to projections

## Try to get monthly projected values for the same station - have to use bounding box instead

In [22]:
%%time

# request takes a long time - testing on 2026-10-13
# 3 min 20 s

# https://api.weather.gc.ca/collections/climate:dcs:projected:monthly:absolute (from CMIP5)
# e.g. the request looks like: https://api.weather.gc.ca/collections/climate:dcs:projected:monthly:absolute/coverage?lang=en-CA&f=json&bbox=-123.4,48.5,-123.35,48.6

# OR is it https://api.weather.gc.ca/collections/climate:candcsu6:projected:monthly:absolute (from CMIP6)
# e.g. the request looks like: https://api.weather.gc.ca/collections/climate:candcsu6:projected:monthly:absolute/coverage?lang=en-CA&f=json&bbox=-123.4,48.5,-123.35,48.6

# try it out
# Compose the request
host = "https://api.weather.gc.ca/"
api = urljoin(host, "collections/climate:candcsu6:projected:monthly:absolute/coverage")
query = {
    #"DATE_CALCULATED" : "2026-10-05",
    #"datetime": "2026-10-05", # all items from 2026-10-05 onward
    #"datetime": "1840-03-01 00:00:00/2021-06-02 00:00:00",
    "bbox": "-123.2,49.1,-123.1,49.2",# minX, minY, maxX, maxY, # YVR is -123.181944	49.195
    #"STN_ID": "889", # Vancouver Intl A # cannot use station name
    "f": "json"
}
url = f"{api}?{urlencode(query)}"


# Use pandas to convert the csv output to a data frame.
with urlopen(url=str(url)) as req:
    df = pd.read_json(req)
df.sort_index().head()

CPU times: user 5 ms, sys: 4.06 ms, total: 9.06 ms
Wall time: 12.5 s


,type,domain,parameters,ranges
AirTemp,Coverage,NaN,"{'type': 'Parameter', 'description': {'en': 'M...","{'type': 'NdArray', 'dataType': 'float', 'axis..."
axes,Coverage,"{'x': {'start': -123.125000715, 'stop': -123.1...",NaN,NaN
domainType,Coverage,Grid,NaN,NaN
referencing,Coverage,"[{'coordinates': ['x', 'y'], 'system': {'type'...",NaN,NaN
type,Coverage,Domain,NaN,NaN


## Can I filter down to get specific year and scenarios?

#### ssp245 50 percentile

In [23]:
%%time

# request takes a long time - testing on 2026-10-13
# 33 sec

# https://api.weather.gc.ca/collections/climate:dcs:projected:monthly:absolute (from CMIP5)
# e.g. the request looks like: https://api.weather.gc.ca/collections/climate:dcs:projected:monthly:absolute/coverage?lang=en-CA&f=json&bbox=-123.4,48.5,-123.35,48.6

# OR is it https://api.weather.gc.ca/collections/climate:candcsu6:projected:monthly:absolute (from CMIP6)
# e.g. the request looks like: https://api.weather.gc.ca/collections/climate:candcsu6:projected:monthly:absolute/coverage?lang=en-CA&f=json&bbox=-123.4,48.5,-123.35,48.6

# try it out
# Compose the request
host = "https://api.weather.gc.ca/"
api = urljoin(host, "collections/climate:candcsu6:projected:monthly:absolute/coverage")
query1 = {
    "bbox": "-123.2,49.1,-123.1,49.2",# minX, minY, maxX, maxY, # YVR is -123.181944	49.195
    "datetime":"2050",
    "subset": "scenario(SSP245)", #percentile(50),scenario('SSP245')",
    "f": "json"
}
# because we have to pass duplicate query parameters for subset
query2 = {
    "subset": "percentile(50)"
}
url = f"{api}?{urlencode(query1)}&{urlencode(query2)}"
print(url)


# Use pandas to convert the csv output to a data frame.
with urlopen(url=str(url)) as req:
    data = json.load(req)

#print(data)

https://api.weather.gc.ca/collections/climate:candcsu6:projected:monthly:absolute/coverage?bbox=-123.2%2C49.1%2C-123.1%2C49.2&datetime=2050&subset=scenario%28SSP245%29&f=json&subset=percentile%2850%29
CPU times: user 3.1 ms, sys: 3.66 ms, total: 6.76 ms
Wall time: 13.1 s


In [24]:
# normalize the json output

norm_df = json_normalize(data)
norm_df

,type,domain.type,domain.domainType,domain.axes.x.start,domain.axes.x.stop,domain.axes.x.num,domain.axes.y.start,domain.axes.y.stop,domain.axes.y.num,domain.axes.t.values,...,parameters.AirTemp.type,parameters.AirTemp.description.en,parameters.AirTemp.unit.symbol,parameters.AirTemp.observedProperty.id,parameters.AirTemp.observedProperty.label.en,ranges.AirTemp.type,ranges.AirTemp.dataType,ranges.AirTemp.axisNames,ranges.AirTemp.shape,ranges.AirTemp.values
0,Coverage,Domain,Grid,-123.125001,-123.125001,1,49.125,49.125,1,"[2050-01, 2050-02, 2050-03, 2050-04, 2050-05, ...",...,Parameter,Mean daily mean temperature,°C,AirTemp,Mean daily mean temperature,NdArray,float,"[y, x, t]","[1, 1, 12]","[5.561248779296875, 6.811981201171875, 8.36407..."


#### ssp585 50 percentile

In [28]:
%%time

# request takes a long time - testing on 2026-10-13
# 33 sec

# https://api.weather.gc.ca/collections/climate:dcs:projected:monthly:absolute (from CMIP5)
# e.g. the request looks like: https://api.weather.gc.ca/collections/climate:dcs:projected:monthly:absolute/coverage?lang=en-CA&f=json&bbox=-123.4,48.5,-123.35,48.6

# OR is it https://api.weather.gc.ca/collections/climate:candcsu6:projected:monthly:absolute (from CMIP6)
# e.g. the request looks like: https://api.weather.gc.ca/collections/climate:candcsu6:projected:monthly:absolute/coverage?lang=en-CA&f=json&bbox=-123.4,48.5,-123.35,48.6

# try it out
# Compose the request
host = "https://api.weather.gc.ca/"
api = urljoin(host, "collections/climate:candcsu6:projected:monthly:absolute/coverage")
query1 = {
    "bbox": "-123.2,49.1,-123.1,49.2",# minX, minY, maxX, maxY, # YVR is -123.181944	49.195
    "datetime":"2050",
    "subset": "scenario(SSP585)", #percentile(50),scenario('SSP245')",
    "f": "json"
}
# because we have to pass duplicate query parameters for subset
query2 = {
    "subset": "percentile(50)"
}
url = f"{api}?{urlencode(query1)}&{urlencode(query2)}"
print(url)


# Use pandas to convert the csv output to a data frame.
with urlopen(url=str(url)) as req:
    data = json.load(req)

#print(data)

https://api.weather.gc.ca/collections/climate:candcsu6:projected:monthly:absolute/coverage?bbox=-123.2%2C49.1%2C-123.1%2C49.2&datetime=2050&subset=scenario%28SSP585%29&f=json&subset=percentile%2850%29
CPU times: user 2.34 ms, sys: 2.27 ms, total: 4.61 ms
Wall time: 14 s


In [29]:
# normalize the json output

norm_df2 = json_normalize(data)
norm_df2

,type,domain.type,domain.domainType,domain.axes.x.start,domain.axes.x.stop,domain.axes.x.num,domain.axes.y.start,domain.axes.y.stop,domain.axes.y.num,domain.axes.t.values,...,parameters.AirTemp.type,parameters.AirTemp.description.en,parameters.AirTemp.unit.symbol,parameters.AirTemp.observedProperty.id,parameters.AirTemp.observedProperty.label.en,ranges.AirTemp.type,ranges.AirTemp.dataType,ranges.AirTemp.axisNames,ranges.AirTemp.shape,ranges.AirTemp.values
0,Coverage,Domain,Grid,-123.125001,-123.125001,1,49.125,49.125,1,"[2050-01, 2050-02, 2050-03, 2050-04, 2050-05, ...",...,Parameter,Mean daily mean temperature,°C,AirTemp,Mean daily mean temperature,NdArray,float,"[y, x, t]","[1, 1, 12]","[5.716583251953125, 7.144500732421875, 8.93386..."


In [30]:
df_proj = pd.DataFrame({'YearMonth': norm_df['domain.axes.t.values'][0], 
                       'MeanTempC_ssp245': norm_df['ranges.AirTemp.values'][0][0:12],
                        'MeanTempC_ssp585': norm_df2['ranges.AirTemp.values'][0][0:12]
                      })
df_proj

,YearMonth,MeanTempC_ssp245,MeanTempC_ssp585
0,2050-01,5.561249,5.716583
1,2050-02,6.811981,7.144501
2,2050-03,8.364075,8.933868
3,2050-04,11.228882,11.389984
4,2050-05,14.906769,14.768829
5,2050-06,17.472351,18.204620
6,2050-07,20.255493,21.157684
7,2050-08,20.427521,21.324738
8,2050-09,17.389099,17.514008
9,2050-10,12.272003,13.512573


In [31]:
#df_proj.query("YearMonth.str.contains('2050')")

## OK those appear to be the projections in absolute temp

### Play around with the linear extrapolation idea

In [33]:
# parameters
month = 10
proj_yr = 2050
yr_month = str(proj_yr)+'-'+str(month)

# tm norm over 1981-2010
tm_norm = df_norms.query("E_NORMAL_ELEMENT_NAME.str.contains('Mean daily temperature') & (MONTH == @month)")['VALUE'].values[0]

# projected value for 
tm_proj_ssp245 = df_proj.query("YearMonth == @yr_month")['MeanTempC_ssp245'].values[0]
tm_proj_ssp585 = df_proj.query("YearMonth == @yr_month")['MeanTempC_ssp585'].values[0]

# trend in temp rise
trend_ssp245 = tm_proj_ssp245/tm_norm
trend_ssp585 = tm_proj_ssp585/tm_norm
print("ssp245",trend_ssp245, tm_norm, tm_proj_ssp245, yr_month)
print("ssp585",trend_ssp585, tm_norm, tm_proj_ssp585, yr_month)



ssp245 1.1926144969706634 10.29 12.272003173828125 2050-10
ssp585 1.3131752421950924 10.29 13.5125732421875 2050-10


### The trend values are the $\beta$ we can use to multiply a temp in Oct

In [34]:
# e.g.
1.19*16

19.04

## Get temps for October and test linear scaling

### Turns out that station 889 doesn't have mean temp data now so need the other station (which is the one that DIDN'T have Normals)

In [8]:
# Compose the request
host = "https://api.weather.gc.ca/"
api = urljoin(host, "collections/climate-daily/items/")
query = {
    "datetime": "2026-10-01/..", # all items from 2026-10-05 onward
    #"datetime": "1840-03-01 00:00:00/2021-06-02 00:00:00",
    #"bbox": "-123.25,49,-123.0,49.25",# minX, minY, maxX, maxY
    #"STN_ID": "889", # Vancouver Intl A
    "STN_ID": "51442", # the other Vancouver Intl A
    "sortby": "LOCAL_DATE",
    "f": "csv"
}
url = f"{api}?{urlencode(query)}"


# Use geopandas to convert the json output to a GeoDataFrame.
with urlopen(url=str(url)) as req:
    df = pd.read_csv(req)
df.sort_index().head(20)

,x,y,STATION_NAME,CLIMATE_IDENTIFIER,ID,LOCAL_DATE,PROVINCE_CODE,LOCAL_YEAR,LOCAL_MONTH,LOCAL_DAY,...,DIRECTION_MAX_GUST_FLAG,SPEED_MAX_GUST,SPEED_MAX_GUST_FLAG,COOLING_DEGREE_DAYS,COOLING_DEGREE_DAYS_FLAG,HEATING_DEGREE_DAYS,HEATING_DEGREE_DAYS_FLAG,MIN_REL_HUMIDITY,MIN_REL_HUMIDITY_FLAG,STN_ID
0,-123.183889,49.194722,VANCOUVER INTL A,1108395,1108395.2026.10.1,2026-10-01 00:00:00,BC,2026,10,1,...,M,NaN,M,0.0,NaN,5.1,NaN,81.0,NaN,51442
1,-123.183889,49.194722,VANCOUVER INTL A,1108395,1108395.2026.10.2,2026-10-02 00:00:00,BC,2026,10,2,...,M,NaN,M,0.0,NaN,2.7,NaN,87.0,NaN,51442
2,-123.183889,49.194722,VANCOUVER INTL A,1108395,1108395.2026.10.3,2026-10-03 00:00:00,BC,2026,10,3,...,M,NaN,M,0.0,NaN,3.4,NaN,89.0,NaN,51442
3,-123.183889,49.194722,VANCOUVER INTL A,1108395,1108395.2026.10.4,2026-10-04 00:00:00,BC,2026,10,4,...,M,NaN,M,0.0,NaN,5.4,NaN,94.0,NaN,51442
4,-123.183889,49.194722,VANCOUVER INTL A,1108395,1108395.2026.10.5,2026-10-05 00:00:00,BC,2026,10,5,...,M,NaN,M,0.0,NaN,5.4,NaN,92.0,NaN,51442
5,-123.183889,49.194722,VANCOUVER INTL A,1108395,1108395.2026.10.6,2026-10-06 00:00:00,BC,2026,10,6,...,M,NaN,M,0.0,NaN,4.7,NaN,95.0,NaN,51442
6,-123.183889,49.194722,VANCOUVER INTL A,1108395,1108395.2026.10.7,2026-10-07 00:00:00,BC,2026,10,7,...,M,NaN,M,0.0,NaN,5.4,NaN,94.0,NaN,51442
7,-123.183889,49.194722,VANCOUVER INTL A,1108395,1108395.2026.10.8,2026-10-08 00:00:00,BC,2026,10,8,...,M,NaN,M,0.0,NaN,3.9,NaN,76.0,NaN,51442
8,-123.183889,49.194722,VANCOUVER INTL A,1108395,1108395.2026.10.9,2026-10-09 00:00:00,BC,2026,10,9,...,NaN,71.0,NaN,0.0,NaN,5.2,NaN,59.0,NaN,51442
9,-123.183889,49.194722,VANCOUVER INTL A,1108395,1108395.2026.10.10,2026-10-10 00:00:00,BC,2026,10,10,...,NaN,37.0,NaN,0.0,NaN,8.6,NaN,73.0,NaN,51442


In [11]:
#df.columns

In [13]:
df_red = df[['x', 'y', 'STATION_NAME', 'ID', 'LOCAL_DATE',
       'PROVINCE_CODE', 'LOCAL_YEAR', 'LOCAL_MONTH', 'LOCAL_DAY',
       'MEAN_TEMPERATURE', 'MIN_TEMPERATURE','MAX_TEMPERATURE',]]
df_red

,x,y,STATION_NAME,ID,LOCAL_DATE,PROVINCE_CODE,LOCAL_YEAR,LOCAL_MONTH,LOCAL_DAY,MEAN_TEMPERATURE,MIN_TEMPERATURE,MAX_TEMPERATURE
0,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.1,2026-10-01 00:00:00,BC,2026,10,1,12.9,10.7,15.1
1,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.2,2026-10-02 00:00:00,BC,2026,10,2,15.3,12.7,17.9
2,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.3,2026-10-03 00:00:00,BC,2026,10,3,14.6,11.9,17.3
3,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.4,2026-10-04 00:00:00,BC,2026,10,4,12.6,8.7,16.4
4,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.5,2026-10-05 00:00:00,BC,2026,10,5,12.6,8.7,16.4
5,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.6,2026-10-06 00:00:00,BC,2026,10,6,13.3,9.3,17.3
6,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.7,2026-10-07 00:00:00,BC,2026,10,7,12.6,9.6,15.6
7,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.8,2026-10-08 00:00:00,BC,2026,10,8,14.1,9.3,18.8
8,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.9,2026-10-09 00:00:00,BC,2026,10,9,12.8,10.4,15.1
9,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.10,2026-10-10 00:00:00,BC,2026,10,10,9.4,4.6,14.2


### Add projected 2050 temp column

In [35]:
df_red['Proj2050ssp245'] = df_red['MEAN_TEMPERATURE']*trend_ssp245
df_red['Proj2050ssp585'] = df_red['MEAN_TEMPERATURE']*trend_ssp585
df_red

,x,y,STATION_NAME,ID,LOCAL_DATE,PROVINCE_CODE,LOCAL_YEAR,LOCAL_MONTH,LOCAL_DAY,MEAN_TEMPERATURE,MIN_TEMPERATURE,MAX_TEMPERATURE,Proj2050ssp245,Proj2050ssp585
0,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.1,2026-10-01 00:00:00,BC,2026,10,1,12.9,10.7,15.1,15.384727,16.939961
1,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.2,2026-10-02 00:00:00,BC,2026,10,2,15.3,12.7,17.9,18.247002,20.091581
2,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.3,2026-10-03 00:00:00,BC,2026,10,3,14.6,11.9,17.3,17.412172,19.172359
3,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.4,2026-10-04 00:00:00,BC,2026,10,4,12.6,8.7,16.4,15.026943,16.546008
4,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.5,2026-10-05 00:00:00,BC,2026,10,5,12.6,8.7,16.4,15.026943,16.546008
5,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.6,2026-10-06 00:00:00,BC,2026,10,6,13.3,9.3,17.3,15.861773,17.465231
6,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.7,2026-10-07 00:00:00,BC,2026,10,7,12.6,9.6,15.6,15.026943,16.546008
7,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.8,2026-10-08 00:00:00,BC,2026,10,8,14.1,9.3,18.8,16.815864,18.515771
8,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.9,2026-10-09 00:00:00,BC,2026,10,9,12.8,10.4,15.1,15.265466,16.808643
9,-123.183889,49.194722,VANCOUVER INTL A,1108395.2026.10.10,2026-10-10 00:00:00,BC,2026,10,10,9.4,4.6,14.2,11.210576,12.343847


## Sweet! Projected 2050 temperatures

## The below isn't quite working right?

### can get it as NetCDF? and then convert to Pandas?

In [43]:
%%time

# request takes a long time - testing on 2026-10-13

# https://api.weather.gc.ca/collections/climate:dcs:projected:monthly:absolute (from CMIP5)
# e.g. the request looks like: https://api.weather.gc.ca/collections/climate:dcs:projected:monthly:absolute/coverage?lang=en-CA&f=json&bbox=-123.4,48.5,-123.35,48.6

# OR is it https://api.weather.gc.ca/collections/climate:candcsu6:projected:monthly:absolute (from CMIP6)
# e.g. the request looks like: https://api.weather.gc.ca/collections/climate:candcsu6:projected:monthly:absolute/coverage?lang=en-CA&f=json&bbox=-123.4,48.5,-123.35,48.6

# try it out
# Compose the request
host = "https://api.weather.gc.ca/"
api = urljoin(host, "collections/climate:candcsu6:projected:monthly:absolute/coverage")
query = {
    #"DATE_CALCULATED" : "2026-10-05",
    #"datetime": "2026-10-05", # all items from 2026-10-05 onward
    #"datetime": "1840-03-01 00:00:00/2021-06-02 00:00:00",
    "bbox": "-123.2,49,-123.1,49.2",# minX, minY, maxX, maxY, # YVR is -123.181944	49.195
    #"STN_ID": "889", # Vancouver Intl A # cannot use station name
    #"properties": "VALUE",
    #"sortby": "LOCAL_DATE",
    "f": "NetCDF"
}
url = f"{api}?{urlencode(query)}"
print(url)

# Use NetCDF Dataset function xarray to open the NetCDF output
data = requests.get(url).content

#da = xr.open_dataarray(data)
#da
ds = xr.open_dataset(data)
ds
    


https://api.weather.gc.ca/collections/climate:candcsu6:projected:monthly:absolute/coverage?bbox=-123.2%2C49%2C-123.1%2C49.2&f=NetCDF
CPU times: user 35.7 ms, sys: 33.9 ms, total: 69.6 ms
Wall time: 24min 8s


KeyboardInterrupt: 

# OK that appears to be the projections but which SSP? just says it is the 50th percentile of ensemble... I think this is if I open it as a dataarray it only gets one scenario. Should open as dataset?

# Note for projections, I can also download the netcdf files per scenario from, e.g. https://dd.weather.gc.ca/today/climate/candcsu6/10km/scenarios/ssp245/ but the files are really big when they are monthly projections (like 450 mb netcdf files)